# Application D — Two short student exercises

Practice fitting and using scikit-learn models. All data are generated below; no files or downloads are needed.

**Instructions:** Run cells in order. At each `TODO`, replace the `raise NotImplementedError(...)`
line with **one or two lines of code**, following the comments. There are five gaps in total.
The error is an intentional reminder that a gap remains. Keep the supplied data and evaluation code.

**Reminder:** `X` is a table of input features, and `y` contains the targets.
Creating a model sets its options; `.fit(X_train, y_train)` learns its parameters.
`.predict(X_test)` makes predictions without changing the fitted model.
For classification, `.predict_proba(X_test)[:, 1]` gives probabilities of class 1.

Each exercise reserves 20% of its rows for testing. Model settings and the classification
threshold are fixed in advance; use the test results for evaluation, not to tune them.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score, confusion_matrix

## 1. Linear regression — predict request latency

**Goal:** Predict latency in milliseconds from payload size in megabytes (MB) and queue length
(number of waiting requests). The generated data follow

$$y=15+3\,\text{payload}+2\,\text{queue}+\varepsilon,\qquad
\varepsilon\sim\mathcal N(0,3^2).$$

The random noise has mean zero and standard deviation 3 milliseconds. Fit a model of the form
$\hat y=b+w_1\,\text{payload}+w_2\,\text{queue}$, where $b$ is the intercept and $w_1,w_2$ are learned slopes.
The split and data generation are supplied.

In [ ]:
rng_linear = np.random.default_rng(41)
n_linear = 400
X_linear = pd.DataFrame({
    "payload": rng_linear.uniform(0.5, 10, n_linear),
    "queue": rng_linear.poisson(3, n_linear),
})
y_linear = (15 + 3 * X_linear["payload"] + 2 * X_linear["queue"]
            + rng_linear.normal(0, 3, n_linear))

X_linear_train, X_linear_test, y_linear_train, y_linear_test = train_test_split(
    X_linear, y_linear, test_size=0.20, random_state=23
)
print("Training rows:", len(X_linear_train), "Test rows:", len(X_linear_test))
display(X_linear_train.head())

### Fit the model

Create a `LinearRegression` object named `linear_model`, then fit it using the training data only.

In [ ]:
# TODO 1 (two lines): create linear_model with LinearRegression().
# Then call its .fit() method with X_linear_train and y_linear_train.
raise NotImplementedError("TODO 1: create and fit the linear regression model.")

### Predict and evaluate

Store the test predictions in `latency_prediction`. Then run the supplied evaluation.
**Mean absolute error (MAE)** is the average absolute error. **Root mean squared error (RMSE)**
squares errors, averages them, and takes the square root, giving more weight to large errors.
Both are measured in milliseconds; smaller is better. The dashed line marks perfect predictions.

In [ ]:
# TODO 2 (one line): call linear_model.predict() with X_linear_test.
# Store the returned predictions in latency_prediction.
raise NotImplementedError("TODO 2: predict test latency.")

In [ ]:
print("Fitted intercept:", round(linear_model.intercept_, 3))
display(pd.Series(linear_model.coef_, index=X_linear.columns, name="Fitted slope"))
print("Test MAE (ms):", round(mean_absolute_error(y_linear_test, latency_prediction), 3))
print("Test RMSE (ms):", round(np.sqrt(mean_squared_error(y_linear_test, latency_prediction)), 3))

plt.figure(figsize=(5, 4))
plt.scatter(y_linear_test, latency_prediction, alpha=0.7)
limits = [min(y_linear_test.min(), latency_prediction.min()),
          max(y_linear_test.max(), latency_prediction.max())]
plt.plot(limits, limits, "--", color="black", label="Perfect prediction")
plt.xlabel("Observed latency (ms)")
plt.ylabel("Predicted latency (ms)")
plt.legend()
plt.tight_layout()
plt.show()

**Explain briefly:** Are the learned intercept and slopes close to 15, 3, and 2?
Why should you expect nonzero prediction errors even with the correct model form?

*Your answer:*

## 2. Logistic regression — predict a budget violation

**Goal:** Estimate the probability that a request exceeds a latency budget.
The label is $y=1$ for a violation and $y=0$ otherwise. This is a new simulated data set;
its labels are generated directly, independently of the first exercise's latencies:

$$z=-4+0.5\,\text{payload}+0.35\,\text{queue},\qquad
p=\frac{1}{1+e^{-z}},\qquad y\sim\operatorname{Bernoulli}(p).$$

The **sigmoid** converts the score $z$ into a probability $p$. A Bernoulli draw is 1 with
probability $p$ and 0 otherwise. The true probabilities are used only to generate labels;
they are not model inputs. `stratify=y_logistic` keeps class proportions approximately
similar in the training and test sets.

In [ ]:
rng_logistic = np.random.default_rng(42)
n_logistic = 800
X_logistic = pd.DataFrame({
    "payload": rng_logistic.uniform(0.5, 10, n_logistic),
    "queue": rng_logistic.poisson(3, n_logistic),
})
true_score = -4 + 0.5 * X_logistic["payload"] + 0.35 * X_logistic["queue"]
true_probability = 1 / (1 + np.exp(-true_score))
y_logistic = rng_logistic.binomial(1, true_probability)

X_logistic_train, X_logistic_test, y_logistic_train, y_logistic_test = train_test_split(
    X_logistic, y_logistic, test_size=0.20, random_state=23, stratify=y_logistic
)
print("Training rows:", len(X_logistic_train), "Test rows:", len(X_logistic_test))
print("Training violation fraction:", round(y_logistic_train.mean(), 3))

### Fit the classifier

Create `logistic_model` using `LogisticRegression(max_iter=1000)`, then fit it on the training data.
`max_iter` limits the number of optimization iterations. Keep the other model options at their defaults.

In [ ]:
# TODO 3 (two lines): create logistic_model with LogisticRegression(max_iter=1000).
# Then call its .fit() method with X_logistic_train and y_logistic_train.
raise NotImplementedError("TODO 3: create and fit the logistic regression model.")

### From probabilities to decisions

First calculate violation probabilities for the test requests. For our labels 0 and 1,
`predict_proba` returns probabilities in that order, so `[:, 1]` selects the second column.
Then use the fixed threshold **0.5**: predict a violation if its probability is at least 0.5.
This is the usual equal-error-cost cutoff; it is not a claim that real alarm costs are equal.

In [ ]:
# TODO 4 (one line): call logistic_model.predict_proba() with X_logistic_test.
# Select column 1 using [:, 1] and store it in violation_probability.
raise NotImplementedError("TODO 4: compute test violation probabilities.")

In [ ]:
# TODO 5 (one line): compare violation_probability with 0.5 using >=.
# Convert the Boolean array to 0/1 using .astype(int); store it in violation_prediction.
raise NotImplementedError("TODO 5: convert probabilities to class predictions.")

### Evaluate the decisions

**Accuracy** is the fraction of correctly classified test requests.
In the confusion matrix, rows are actual labels and columns are predicted labels:
upper-right counts **false alarms**; lower-left counts **missed violations**.
The first table helps distinguish probabilities from the resulting 0/1 decisions.

In [ ]:
display(pd.DataFrame({
    "Actual label": y_logistic_test,
    "Predicted probability": violation_probability,
    "Predicted label": violation_prediction,
}).head(8))
print("Test accuracy:", round(accuracy_score(y_logistic_test, violation_prediction), 3))

matrix = confusion_matrix(y_logistic_test, violation_prediction, labels=[0, 1])
display(pd.DataFrame(matrix, index=["Actual 0", "Actual 1"],
                     columns=["Predicted 0", "Predicted 1"]))

**Explain briefly:** How many false alarms and missed violations are there?
What class is predicted for probability 0.40? Does that probability guarantee the actual outcome?

*Your answer:*